# Map CCLE Cell Line Embeddings to ALL Dataset h5ad Files

Add 300-dim CCLE PCA embeddings to `adata.uns["cell_line_ccle_embeddings"]` for every dataset 
in `drug.yaml`. Keeps the existing one-hot `cell_line_embeddings` intact.

**CCLE source**: `cell_line_embedding_full_ccle_300_scaled.csv` (1406 cell lines x 300 PCA dims)

**Name mapping strategy**:
- Tahoe uses CVCL IDs (e.g., `CVCL_0023`) → manual CVCL-to-common-name map → CCLE match
- Other datasets (sciplex, combosciplex, etc.) use common names directly → normalize and match

In [5]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import yaml

CCLE_PATH = '/lustre/groups/ml01/projects/super_rad_project/cell_line_embeddings/cell_line_embedding_full_ccle_300_scaled.csv'
DATA_DIR = '/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/unipert'
DRUG_YAML = '/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/CellFlow2/experiments/config/datasets/drug.yaml'

# Key name for the new CCLE embeddings
CCLE_KEY = 'cell_line_ccle_embeddings'

# Known CVCL -> common name mapping (for Tahoe and any other CVCL-based datasets)
CVCL_TO_COMMON = {
    'CVCL_0023': 'A549',
    'CVCL_1724': 'SW48',
    'CVCL_0099': 'SNU-1',
    'CVCL_1239': 'H4',
    'CVCL_0152': 'AsPC-1',
    'CVCL_1285': 'HOP-62',
    'CVCL_0480': 'PANC-1',
}

# Load all dataset paths from drug.yaml
with open(DRUG_YAML) as f:
    drug_config = yaml.safe_load(f)

ALL_FILES = {name: info['path'] for name, info in drug_config.items()}
print(f'Datasets from drug.yaml: {len(ALL_FILES)}')
for name, path in ALL_FILES.items():
    exists = '✓' if os.path.exists(path) else '✗'
    print(f'  {exists} {name}: {os.path.basename(path)}')

Datasets from drug.yaml: 13
  ✓ combosciplex: combosciplex_se.h5ad
  ✓ tahoe_a549: tahoe_a549_w_emb.h5ad
  ✓ tahoe_aspc_1: tahoe_aspc_1_w_emb.h5ad
  ✓ tahoe_h4: tahoe_h4_w_emb.h5ad
  ✓ tahoe_hop62: tahoe_hop62_w_emb.h5ad
  ✓ tahoe_panc_1: tahoe_panc_1_w_emb.h5ad
  ✓ tahoe_snu_1: tahoe_snu_1_w_emb.h5ad
  ✓ tahoe_sw48: tahoe_sw48_w_emb.h5ad
  ✓ zhao21: zhao21_w_emb.h5ad
  ✓ aissa21: aissa21_w_emb.h5ad
  ✓ sciplex2: srivatsan20_sciplex2_w_emb.h5ad
  ✓ sciplex3: srivatsan20_sciplex3_w_emb.h5ad
  ✓ sciplex4: srivatsan20_sciplex4_w_emb.h5ad


## 1. Load CCLE Embeddings and Build Name Lookup

In [6]:
# Load CCLE embeddings (1406 cell lines x 300 dims)
ccle_df = pd.read_csv(CCLE_PATH, index_col=0)
print(f'CCLE embeddings: {ccle_df.shape[0]} cell lines x {ccle_df.shape[1]} dims')

# Build normalized lookup: lowercase, no hyphens/spaces -> (original CCLE name, embedding)
def normalize_name(name):
    return str(name).lower().replace('-', '').replace(' ', '').replace('_', '')

ccle_lookup = {}
for ccle_name in ccle_df.index:
    norm = normalize_name(ccle_name)
    ccle_lookup[norm] = {
        'ccle_name': ccle_name,
        'embedding': ccle_df.loc[ccle_name].values.astype(np.float32),
    }

print(f'Normalized lookup: {len(ccle_lookup)} entries')
print(f'Embedding dim: {ccle_df.shape[1]}')

CCLE embeddings: 1406 cell lines x 300 dims
Normalized lookup: 1406 entries
Embedding dim: 300


## 2. Scan ALL Datasets and Preview Cell Line Matching

In [7]:
# Scan all datasets for cell line names and match to CCLE
all_mappings = {}  # {dataset_name: {cell_line_in_obs: ccle_name_or_None}}

for ds_name, path in ALL_FILES.items():
    if not os.path.exists(path):
        print(f'\n{ds_name}: FILE NOT FOUND, skipping')
        continue
    
    adata = ad.read_h5ad(path, backed='r')
    cls = adata.obs['cell_line'].unique().tolist()
    del adata
    
    mapping = {}
    for cl in cls:
        emb, matched = get_ccle_embedding(cl, CVCL_TO_COMMON, ccle_lookup)
        mapping[cl] = matched if matched else None
    
    all_mappings[ds_name] = mapping
    
    n_matched = sum(1 for v in mapping.values() if v is not None)
    n_total = len(mapping)
    status = '✓ ALL' if n_matched == n_total else f'{n_matched}/{n_total}'
    
    print(f'\n{ds_name} ({status}):')
    for cl, matched in mapping.items():
        if matched:
            print(f'  {cl} -> {matched} ✓')
        else:
            print(f'  {cl} -> ??? ✗ (not in CCLE)')

# Summary
print(f'\n{"=" * 60}')
print('SUMMARY:')
total_cls = set()
matched_cls = set()
for ds_name, mapping in all_mappings.items():
    for cl, matched in mapping.items():
        total_cls.add(cl)
        if matched:
            matched_cls.add(cl)

unmatched = total_cls - matched_cls
print(f'  Total unique cell lines across all datasets: {len(total_cls)}')
print(f'  Matched to CCLE: {len(matched_cls)}')
print(f'  Unmatched: {len(unmatched)}')
if unmatched:
    print(f'  Unmatched names: {sorted(unmatched)}')


combosciplex (✓ ALL):
  A549 -> A549 ✓

tahoe_a549 (✓ ALL):
  CVCL_0023 -> A549 ✓

tahoe_aspc_1 (✓ ALL):
  CVCL_0152 -> ASPC1 ✓

tahoe_h4 (✓ ALL):
  CVCL_1239 -> H4 ✓

tahoe_hop62 (✓ ALL):
  CVCL_1285 -> HOP62 ✓

tahoe_panc_1 (✓ ALL):
  CVCL_0480 -> PANC1 ✓

tahoe_snu_1 (✓ ALL):
  CVCL_0099 -> SNU1 ✓

tahoe_sw48 (✓ ALL):
  CVCL_1724 -> SW48 ✓

zhao21 (0/1):
  na -> ??? ✗ (not in CCLE)

aissa21 (✓ ALL):
  PC-9 -> PC9 ✓


/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



sciplex2 (✓ ALL):
  A549 -> A549 ✓

sciplex3 (3/4):
  MCF7 -> MCF7 ✓
  A549 -> A549 ✓
  K-562 -> K562 ✓
  NA -> ??? ✗ (not in CCLE)

sciplex4 (2/3):
  A549 -> A549 ✓
  MCF7 -> MCF7 ✓
  NA -> ??? ✗ (not in CCLE)

SUMMARY:
  Total unique cell lines across all datasets: 13
  Matched to CCLE: 11
  Unmatched: 2
  Unmatched names: ['NA', 'na']


/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


## 3. Add CCLE Embeddings to ALL h5ad Files

For unmatched cell lines, a zero vector is stored (the model's masking handles this).

In [8]:
CCLE_DIM = ccle_df.shape[1]  # 300
zero_emb = np.zeros(CCLE_DIM, dtype=np.float32)

for ds_name, path in ALL_FILES.items():
    if not os.path.exists(path):
        print(f'\n{ds_name}: SKIP (file not found)')
        continue
    if ds_name not in all_mappings:
        continue
    
    mapping = all_mappings[ds_name]
    
    print(f'\n{"=" * 60}')
    print(f'Processing: {ds_name} ({os.path.basename(path)})')
    
    # Build CCLE embedding dict using obs cell_line values as keys
    ccle_emb_dict = {}
    for cl, ccle_name in mapping.items():
        if ccle_name is not None:
            emb, _ = get_ccle_embedding(cl, CVCL_TO_COMMON, ccle_lookup)
            ccle_emb_dict[cl] = emb
            print(f'  {cl} -> {ccle_name} (300d)')
        else:
            ccle_emb_dict[cl] = zero_emb.copy()
            print(f'  {cl} -> ZERO (no CCLE match)')
    
    # Load full file, add embeddings, write back
    print(f'  Loading full file...')
    adata = sc.read_h5ad(path)
    
    adata.uns[CCLE_KEY] = ccle_emb_dict
    adata.uns['cell_line_ccle_embedding_dim'] = CCLE_DIM
    
    print(f'  Writing back...')
    adata.write_h5ad(path)
    
    n_real = sum(1 for v in ccle_emb_dict.values() if np.any(v != 0))
    print(f'  Done: {CCLE_KEY} with {n_real}/{len(ccle_emb_dict)} real CCLE embeddings')
    del adata

print(f'\n{"=" * 60}')
print('ALL DONE')


Processing: combosciplex (combosciplex_se.h5ad)
  A549 -> A549 (300d)
  Loading full file...
  Writing back...
  Done: cell_line_ccle_embeddings with 1/1 real CCLE embeddings

Processing: tahoe_a549 (tahoe_a549_w_emb.h5ad)
  CVCL_0023 -> A549 (300d)
  Loading full file...
  Writing back...
  Done: cell_line_ccle_embeddings with 1/1 real CCLE embeddings

Processing: tahoe_aspc_1 (tahoe_aspc_1_w_emb.h5ad)
  CVCL_0152 -> ASPC1 (300d)
  Loading full file...
  Writing back...
  Done: cell_line_ccle_embeddings with 1/1 real CCLE embeddings

Processing: tahoe_h4 (tahoe_h4_w_emb.h5ad)
  CVCL_1239 -> H4 (300d)
  Loading full file...
  Writing back...
  Done: cell_line_ccle_embeddings with 1/1 real CCLE embeddings

Processing: tahoe_hop62 (tahoe_hop62_w_emb.h5ad)
  CVCL_1285 -> HOP62 (300d)
  Loading full file...
  Writing back...
  Done: cell_line_ccle_embeddings with 1/1 real CCLE embeddings

Processing: tahoe_panc_1 (tahoe_panc_1_w_emb.h5ad)
  CVCL_0480 -> PANC1 (300d)
  Loading full file...

/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  Writing back...
  Done: cell_line_ccle_embeddings with 1/1 real CCLE embeddings

Processing: sciplex3 (srivatsan20_sciplex3_w_emb.h5ad)
  MCF7 -> MCF7 (300d)
  A549 -> A549 (300d)
  K-562 -> K562 (300d)
  NA -> ZERO (no CCLE match)
  Loading full file...
  Writing back...
  Done: cell_line_ccle_embeddings with 3/4 real CCLE embeddings

Processing: sciplex4 (srivatsan20_sciplex4_w_emb.h5ad)
  A549 -> A549 (300d)
  MCF7 -> MCF7 (300d)
  NA -> ZERO (no CCLE match)
  Loading full file...


/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  Writing back...
  Done: cell_line_ccle_embeddings with 2/3 real CCLE embeddings

ALL DONE


## 4. Verify

In [9]:
test_file = os.path.join(DATA_DIR, TAHOE_FILES[0])
adata = ad.read_h5ad(test_file, backed='r')

print(f'File: {TAHOE_FILES[0]}')

if CCLE_KEY in adata.uns:
    print(f'\n{CCLE_KEY} (new, 300d CCLE PCA):')
    for cl, emb in adata.uns[CCLE_KEY].items():
        print(f'  {cl}: shape={emb.shape}, values[:5]={emb[:5]}')

if 'cell_line_embeddings' in adata.uns:
    print(f'\ncell_line_embeddings (old, one-hot) still intact:')
    for cl, emb in adata.uns['cell_line_embeddings'].items():
        print(f'  {cl}: shape={emb.shape}')

del adata

File: tahoe_a549_w_emb.h5ad

cell_line_ccle_embeddings (new, 300d CCLE PCA):
  CVCL_0023: shape=(300,), values[:5]=[-0.59626555  0.03409106  0.3015831   0.472836    0.71360195]

cell_line_embeddings (old, one-hot) still intact:
  CVCL_0023: shape=(1,)
